In [ ]:
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, concatenate_datasets, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
def prepare_unified_dataset_v3():
    all_data = []

    # 1. HaluEval - Usually script-free, but add flag just in case
    halu = load_dataset("pminervini/HaluEval", "qa", split="data", trust_remote_code=True)
    for item in halu.select(range(5000)):
        all_data.append({"text": f"Context: {item['knowledge']} | Q: {item['question']} | A: {item['right_answer']}", "label": 0})
        all_data.append({"text": f"Context: {item['knowledge']} | Q: {item['question']} | A: {item['hallucinated_answer']}", "label": 1})

    # 2. TruthfulQA - Uses JSON, usually doesn't need the flag
    tqa = load_dataset("truthful_qa", "generation", split="validation")
    for item in tqa:
        all_data.append({"text": f"Q: {item['question']} | A: {item['best_answer']}", "label": 0})
        all_data.append({"text": f"Q: {item['question']} | A: {item['incorrect_answers'][0]}", "label": 1})

    # 3. FEVER (The 3rd Dataset) - Direct Parquet Loading
    # If standard loading fails, we use the Parquet export branch
    try:
        fever = load_dataset("fever", "v1.0", split="train", trust_remote_code=True)
    except Exception as e:
        print("Standard FEVER load failed, switching to Parquet...")
        fever = load_dataset("parquet", data_files="hf://datasets/fever/fever@refs/convert/parquet/v1.0/train/*.parquet", split="train")

    fever_filtered = fever.filter(lambda x: x["label"] in ["SUPPORTS", "REFUTES"]).shuffle(seed=42).select(range(5000))

    for item in fever_filtered:
        label = 0 if item["label"] == "SUPPORTS" else 1
        all_data.append({"text": f"Fact Check: {item['claim']}", "label": label})

    combined_ds = Dataset.from_list(all_data).shuffle(seed=42)
    return combined_ds.train_test_split(test_size=0.1)

dataset = prepare_unified_dataset_v3()
print(f"Dataset Size: {len(dataset['train'])} rows")
print(f"Labels: {pd.Series(dataset['train']['label']).value_counts()}")

In [ ]:
# Assuming 'dataset' is your current merged dataset
minority_count = 6463 # The count for Label 1

# Separate and Downsample
ds_truth = dataset['train'].filter(lambda x: x['label'] == 0).shuffle(seed=42).select(range(minority_count))
ds_hallu = dataset['train'].filter(lambda x: x['label'] == 1)

# Re-merge to a perfect 12,926 sample dataset
balanced_dataset = concatenate_datasets([ds_truth, ds_hallu]).shuffle(seed=42)

print(f"Balanced Dataset: {len(balanced_dataset)} samples")
print(f"New Split: 50% / 50%")

In [ ]:
# Assuming 'dataset' is your current merged dataset
minority_count = 6463 # The count for Label 1

# Separate and Downsample
ds_truth = dataset['train'].filter(lambda x: x['label'] == 0).shuffle(seed=42).select(range(minority_count))
ds_hallu = dataset['train'].filter(lambda x: x['label'] == 1)

# Re-merge to a perfect 12,926 sample dataset
balanced_dataset = concatenate_datasets([ds_truth, ds_hallu]).shuffle(seed=42)

print(f"Balanced Dataset: {len(balanced_dataset)} samples")
print(f"New Split: 50% / 50%")

In [ ]:
#  Load Model & Tokenizer
MODEL_ID = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)

# Tokenize the Balanced Dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)


tokenized_datasets = balanced_dataset.map(tokenize_function, batched=True)
# Split it for evaluation
tokenized_split = tokenized_datasets.train_test_split(test_size=0.1)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Convert raw scores (logits) into 0 or 1
    predictions = np.argmax(logits, axis=-1)

    # Calculate Accuracy and F1
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions)

    return {
        "accuracy": acc,
        "f1": f1
    }

Training the Discriminator

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback, DataCollatorWithPadding
import numpy as np

# 1. Define the "Saturation" Arguments
training_args = TrainingArguments(
    output_dir="./hallucination_detector_v1",

    # LIMITLESS LOGIC:
    # The model will ONLY stop once it stops improving.
    max_steps=100000,

    eval_strategy="steps",
    eval_steps=100,               # Check for saturation every 100 steps
    save_steps=100,               # Save progress every 100 steps
    logging_steps=50,

    learning_rate=3e-5,           # ModernBERT standard
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    fp16=True,

    load_best_model_at_end=True,
    metric_for_best_model="f1",   # Stop based on F1-score (Truth vs Hallucination balance)
    greater_is_better=True,
    save_total_limit=2,           # Keep only the best 2 models to save disk space
    report_to="tensorboard"
)

# 2. Initialize Trainer with Early Stopping
# patience=5 means: "If F1 doesn't improve for 500 steps (5 checks), EXIT."
early_stop = EarlyStoppingCallback(early_stopping_patience=5)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_split["train"],
    eval_dataset=tokenized_split["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics, # Uses the function you defined in Cell 4
    callbacks=[early_stop]
)
# 3. Start the Engine
print("🚀 Training starting... Watch the TensorBoard tab above for live saturation plots.")
trainer.train()

# 4. Save the Result
trainer.save_model("./hallucination_model_v1")
print("✅ Training complete. Best model saved to ./hallucination_model_v1")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def plot_final_results(trainer, tokenized_test_dataset):
    # 1. Get predictions on the test set
    outputs = trainer.predict(tokenized_test_dataset)
    y_pred = np.argmax(outputs.predictions, axis=-1)
    y_true = outputs.label_ids

    # 2. Create Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] # Normalized for percentage

    # 3. Plotting
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="Blues",
                xticklabels=["Truth (0)", "Hallucination (1)"],
                yticklabels=["Truth (0)", "Hallucination (1)"])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Hallucination Detector: Confusion Matrix (Saturation Point)")
    plt.show()

# Call this after trainer.train()
plot_final_results(trainer, tokenized_split["test"])

Discriminator performance

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def plot_final_results(trainer, tokenized_test_dataset):
    # 1. Get predictions on the test set
    outputs = trainer.predict(tokenized_test_dataset)
    y_pred = np.argmax(outputs.predictions, axis=-1)
    y_true = outputs.label_ids

    # 2. Create Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] # Normalized for percentage

    # 3. Plotting
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="Blues",
                xticklabels=["Truth (0)", "Hallucination (1)"],
                yticklabels=["Truth (0)", "Hallucination (1)"])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Hallucination Detector: Confusion Matrix (Saturation Point)")
    plt.show()

# Call this after trainer.train()
plot_final_results(trainer, tokenized_split["test"])